<h1>Import libraries</h1>

In [1]:
import numpy as np
from pandas import DataFrame
from pathlib import Path

<h1>Set parameters</h1>

In [2]:
RANDOM_SEED = 47
N = 350
THETA_P_LB = 1/3
THETA_P_UB = 1
THETA_INCREMENT = 0.01
F_INCREMENT = 0.01
T_UB = 11
DECIMAL_ROUNDING = 4

<h1>Set random seed</h1>

In [3]:
np.random.seed(RANDOM_SEED)

<h1>Monte Carlo simulation</h1>

<h2>Parameter space</h2> Because a player is choosing the correct answer out of three options, we can bound their ability with $\frac{1}{3} \leq \theta_p \leq 1$. We asuume that the chaser's ability is always greater than or equal to the player's, so we bound that with $\theta_p \leq \theta_p \leq 1$. We'll express the difference between the player's and chaser's ability using the expression $f = \theta_c - \theta_p$. Finally, to represent the three choices of offer (low, middle and high) we'll use the variable $s$ where $s=-1$ corresponds to the player taking the high offer, $s=0$ the middle and $s=1$ the low offer.

<h2>Winning and losing</h2> A player wins where they move to the eigth position on the board. The chaser wins where they get to the same position on the board. The winning margin is the number of spaces between the player and chaser after player answers correctly. The losing margin is the number of spaces between the players and home.

<h2>Simulation</h2> This simulates $N$ head-to-heads for the each unique set of variables in the parameter space and returns the winning margin for each one.

In [4]:
theta_p_values = np.linspace(THETA_P_LB, THETA_P_UB, round((THETA_P_UB-THETA_P_LB)/THETA_INCREMENT)+1)

results = []

for theta_p in theta_p_values:
    
    theta_c_values = np.linspace(theta_p, 1, round((1-theta_p)/F_INCREMENT)+1)
    
    for theta_c in theta_c_values:
        
        for s in [-1, 0, 1]:
            
            for i in range(1, N+1):
                
                # Inital board positions
                P = 3 + s
                C = 0
                
                # Number of rounds
                t = 0 

                while True:
                    t += 1
                    
                    P += np.random.binomial(1, theta_p)                 
        
                    if P == 8: # Player wins 
                        # Winning margin is number of spaces between the player and chaser after correct answer
                        margin = P - C - 1
                        break
                    
                    C += np.random.binomial(1, theta_c)
                    
                    if P == C: # Chaser wins 
                        # Losing margin is number of spaces between the player and home
                        margin = C - 8
                        break
                
                results.append([theta_p, theta_c, s, i, P, C, margin, t])
                            
            if len(results) % 50000 == 0:
                print(f"Completed {len(results)} trials")

Completed 350000 trials
Completed 700000 trials
Completed 1050000 trials
Completed 1400000 trials
Completed 1750000 trials
Completed 2100000 trials
Completed 2450000 trials


<h1>Save results to dataframe</h1>

In [5]:
data = DataFrame(
    results,
    columns=["theta_p", "theta_c", "s", "i", "P", "C", "m", "t"]
)  

<h1>Cleaning results</h1>

<h2>Set player win</h2>

In [6]:
data['player_win'] = np.where(data['m']< 0, 0, 1)

<h2>Round the decimals</h2>

In [7]:
data['theta_p'] = data['theta_p'].round(DECIMAL_ROUNDING)
data['theta_c'] = data['theta_c'].round(DECIMAL_ROUNDING)

<h2>Remove head-to-heads which took too long</h2>

In [8]:
data = data[data['t'] <= 11]

In [9]:
data = data.reset_index(drop=True)

<h1>Viewing results</h1>

In [10]:
data.head()

,theta_p,theta_c,s,i,P,C,m,t,player_win
0,0.3333,0.3333,-1,5,2,2,-6,2,0
1,0.3333,0.3333,-1,6,4,4,-4,7,0
2,0.3333,0.3333,-1,8,3,3,-5,3,0
3,0.3333,0.3333,-1,9,8,1,6,8,1
4,0.3333,0.3333,-1,10,3,3,-5,3,0


<h1>Save results to dataframe</h1>

In [11]:
out_path = Path(f"{Path.cwd().parent}/data/processed/mc_head_to_head_results.csv")
data.to_csv(out_path, index=False)
print("Saved to:", out_path.resolve())  # shows the full path it was saved to

Saved to: /Users/ben/Documents/The Chase/data/processed/mc_head_to_head_results.csv
